In [ ]:
library(ggplot2)
library(behavr)
library(scopr)
library(sleepr)
library(ggetho)
library(plotly)
#library(survival)
library(cowplot)
#library(ggthemes)
library(plotly)
library(data.table)
library(stringi)
library(ggtern)
library(ggpubr)

In [ ]:
#load these folders
REMOTE_DATA_SOURCE <- "ftp://etho-node.lab.gilest.ro/auto_generated_data/ethoscope_results/"
MY_DIR <- "/insecticides/"
setwd(MY_DIR)
# This is the placement of the data in this computer, make sure this is a file where you want it saved!
DATA_DIR <- "/mnt/ethoscope_results"
#This is the placement of the real data, on the NAS
#this is the place were a cache version of the data is stored, once it has been taken from NAS. This make the loading faster on the second time.
CACHE <- "/home/cache"
#This is the query of the experiment (this is the table you made of the data)
METADATA <- "/insecticides/ethoscope_metadata_1ppm.csv"

In [ ]:
#To get the files from the remote source
query <- link_ethoscope_metadata(METADATA,
                                 result_dir = DATA_DIR)

In [ ]:
#This is the magic step, it loads the data to R and applies a function at the same time, in this case, the asleep annotation.
dt <- load_ethoscope(query,
                     reference_hour = 9.0, 
                     FUN = sleep_annotation,
                     velocity_correction_coef = 0.01,
                     cache = CACHE)

In [ ]:
#to include baseline days if there are multiple conditions 
dt[,t:=t+days(xmv(baseline_days))]

In [ ]:
#remove data for dead animals after their death.
dt_curated <- curate_dead_animals(dt)
summary(dt_curated)

In [ ]:
#to add day number, and light phase
dt [,day:=floor(t/days(1))]
dt [,phase:=ifelse(t %% hours(24)>hours(12),"Dark","Light")]
dt [,phase:=factor(phase, levels= c("Light","Dark"))]
gc()

In [ ]:
dt_12hrs<- dt[t>days(0) &t<days(0.5)]

In [ ]:
dt_12hrs <- dt_12hrs[dt_12hrs, meta=T]

In [ ]:
#select for specific compound group
dt_my_compound_12hrs <- dt_12hrs[compound=="my_compound"]

In [ ]:
#make table with only max velocity data by id
dt_my_compound_12hrs_velocity <- dt_my_compound_12hrs[, .(max_velocity), by=id]
print(dt_my_compound_12hrs_velocity)

In [ ]:
#puts the data into a list by fly id 
split.df <- split(dt_my_compound_12hrs_velocity, dt_my_compound_12hrs_velocity$id)

In [ ]:
#splits data for the compound into individual time series (with ascending numbers) csv files which are stored in a folder of your choice. 
#Choose a good folder.
for(i in 1:length(split.df)){
  write.csv(split.df[[i]], paste0("insecticides/mycomp_velocity_files/timeseries_",i, 
                                  ".csv"))
}

In [ ]:
#to remove all empty timeseries in a folder, which will have a file size of 23Kb
## Get vector of all file names
ff <- dir("insecticides/mycomp_velocity_files", recursive=TRUE, full.names=TRUE)
## Extract vector of empty files' names
eff <- ff[file.info(ff)[["size"]]==23]
## Remove empty files
unlink(eff, recursive=TRUE, force=FALSE)